In [1]:
from src.ssunet.datasets import BinomDataset # ValidationDataset removed
from src.ssunet.models import Bit2Bit
from src.ssunet.configs import MasterConfig
from src.ssunet.constants import DEFAULT_CONFIG_PATH


from pathlib import Path

# --- Configuration Loading ---
try:
    config = MasterConfig.from_config(DEFAULT_CONFIG_PATH)
    print(f"Successfully loaded configuration from: {DEFAULT_CONFIG_PATH}")
    print(f"Experiment directory: {config.target_path}")
    print(f"Config copied to: {config.target_path / Path(DEFAULT_CONFIG_PATH).name}")
except FileNotFoundError:
    print(f"ERROR: Default configuration file not found at '{DEFAULT_CONFIG_PATH}'. Please check the path.")
except Exception as e:
    print(f"ERROR loading configuration: {e}")

# --- Data Loading ---
print("Loading data...")
try:
    ssunet_training_data_source = config.path_config.load_data_only()
    print(f"  Training data source loaded. Shape: {ssunet_training_data_source.primary_data.shape}, Type: {type(ssunet_training_data_source.primary_data)}")

    # This validation data source might be used for other purposes (e.g., final evaluation with ground truth)
    # or can be removed if not needed elsewhere in the notebook.
    ssunet_validation_data_source = config.path_config.load_reference_and_ground_truth()
    print(f"  External validation data source loaded (for potential separate evaluation).")
    print(f"    Reference (for input) shape: {ssunet_validation_data_source.primary_data.shape if ssunet_validation_data_source.primary_data is not None else 'N/A'}")
    print(f"    Ground Truth (for target) shape: {ssunet_validation_data_source.secondary_data.shape if ssunet_validation_data_source.secondary_data is not None else 'N/A'}")
except FileNotFoundError as e:
    print(f"ERROR: Data file not found: {e}. Check PATH configuration in your config file.")
except Exception as e:
    print(f"ERROR loading data: {e}")

# --- Dataset Configuration & Creation ---
print("Configuring and creating datasets...")
data_config = config.data_config

training_data = BinomDataset(ssunet_training_data_source, data_config, config.split_params)
print(f"  Training dataset created. Length: {len(training_data)}")

# Create validation dataset using the method from the training_data (BinomDataset) instance
# This will create a BinomIdentityValidationDataset internally, using the training data source.
validation_data = training_data.create_identity_validation_dataset()
print(f"  Validation dataset (identity from training source) created. Length: {len(validation_data)}")

# --- Model Creation ---
print("Initializing model...")
model = Bit2Bit(config.model_config)

# --- Data Loaders ---
print("Creating data loaders...")
training_loader = config.loader_config.loader(training_data)
validation_loader = config.loader_config.loader(validation_data)

# --- Check Input Size (Optional Sanity Check) ---
try:
    # For BinomDataset: batch[0] is target_cdhw, batch[1] is noise_cdhw (model input)
    # For BinomIdentityValidationDataset: batch[0] is original_patch (target), batch[1] is original_patch (model input)
    # The Bit2Bit model expects: y_loss_target = batch[0], x (model_input) = batch[1]
    
    sample_batch_from_training_loader = next(iter(training_loader))
    sample_loss_target_batch = sample_batch_from_training_loader[0]
    sample_model_input_batch = sample_batch_from_training_loader[1]
    
    print(f"  Sample model input batch shape from training_loader: {tuple(sample_model_input_batch.shape)}")
    print(f"  Sample loss target batch shape from training_loader: {tuple(sample_loss_target_batch.shape)}")

    if len(validation_loader) > 0:
        sample_batch_from_validation_loader = next(iter(validation_loader))
        sample_val_loss_target_batch = sample_batch_from_validation_loader[0]
        sample_val_model_input_batch = sample_batch_from_validation_loader[1]
        print(f"  Sample model input batch shape from validation_loader: {tuple(sample_val_model_input_batch.shape)}")
        print(f"  Sample loss target batch shape from validation_loader: {tuple(sample_val_loss_target_batch.shape)}")
        if len(sample_batch_from_validation_loader) > 2:
            sample_val_metrics_target_batch = sample_batch_from_validation_loader[2]
            print(f"  Sample metrics target batch shape from validation_loader: {tuple(sample_val_metrics_target_batch.shape)}")
    else:
        print("  Validation loader is empty or not configured for sample check.")

except StopIteration:
    print("ERROR: Training or Validation loader is empty. Check dataset size and batch_size.")
except Exception as e:
    print(f"Error getting sample batch: {e}")

# --- Model Training ---
print("Starting model training...")
trainer = config.trainer
try:
    trainer.fit(model, training_loader, validation_loader)
    print("Training completed.")
except Exception as e:
    print(f"An error occurred during training or subsequent steps: {e}")

print("Script finished.")

Successfully loaded configuration from: config.yml
Experiment directory: models\20250530_201940_e=120_p=32_d=[0]_test_run_v2_0x32x256x256_skip=1_d=5_sf=32_ds=2at10_f=10.0_z=3_g=8_sd=0_b=tri_a=gelu
Config copied to: models\20250530_201940_e=120_p=32_d=[0]_test_run_v2_0x32x256x256_skip=1_d=5_sf=32_ds=2at10_f=10.0_z=3_g=8_sd=0_b=tri_a=gelu\config.yml
Loading data...
  Training data source loaded. Shape: (3600, 512, 512), Type: <class 'numpy.ndarray'>
  External validation data source loaded (for potential separate evaluation).
    Reference (for input) shape: (255, 512, 512)
    Ground Truth (for target) shape: (255, 512, 512)
Configuring and creating datasets...
  Training dataset created. Length: 3569
  Validation dataset (identity from training source) created. Length: 3569
Initializing model...
Creating data loaders...
  Sample model input batch shape from training_loader: (2, 1, 32, 256, 256)
  Sample loss target batch shape from training_loader: (2, 1, 32, 256, 256)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


  Sample model input batch shape from validation_loader: (2, 1, 32, 256, 256)
  Sample loss target batch shape from validation_loader: (2, 1, 32, 256, 256)
Starting model training...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type                             | Params | Mode 
-------------------------------------------------------------------------
0 | psnr_metric | PeakSignalNoiseRatio             | 0      | train
1 | ssim_metric | StructuralSimilarityIndexMeasure | 0      | train
2 | down_convs  | ModuleList                       | 8.8 M  | train
3 | up_convs    | ModuleList                       | 4.7 M  | train
4 | conv_final  | Sequential                       | 33     | train
-------------------------------------------------------------------------
13.5 M    Trainable params
0         Non-trainable params
13.5 M    Total params
54.156    Total estimated model params size (MB)
86        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=120` reached.
FIT Profiler Report

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Action                                                                                                                                                               	|  Mean duration (s)	|  Num calls      	|  Total time (s) 	|  Percentage %   	|
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Total                                                                                                                         

Training completed.
Script finished.


In [ ]:
trainer.save_checkpoint(config.train_config.default_root_dir / "model.ckpt")


In [ ]:
from src.tools.tools import clear_vram
clear_vram()



In [ ]:
from src.tools.gpuinference import gpu_patch_inference
import numpy as np
from tifffile import imwrite

output = gpu_patch_inference(
    model,
    data.primary_data.astype(np.float32),
    min_overlap=48,
    initial_patch_depth=64,
    device=config.device,
)

imwrite(config.train_config.default_root_dir / "input.tif", data.primary_data)
imwrite(config.train_config.default_root_dir / "inference.tif", output)

In [ ]:
from src.tools.tools import group_metrics
from tifffile import imread


input_new = data.primary_data.astype(np.float32)[:512].copy()
output_new = output[:512].copy()
ground_truth = imread(config.path_config.ground_truth_file)[:512].astype(np.float32).copy()

group_metrics(input_new, output_new, ground_truth, config.train_config.default_root_dir)
